In [1]:
import hashlib, json, os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import serve_stress

PLAIN, RACE = repo / "build/adapters", repo / "build/race"
git = lambda *a, cwd=repo: subprocess.run(["git", *a], cwd=cwd, capture_output=True, text=True).stdout.strip()
sh = lambda *a, **k: subprocess.run(a, check=True, capture_output=True, text=True, **k)
# The plain build writes the corpora; the instrumented build serves. Both from this checkout.
sh("make", "adapters", cwd=repo)
RACE.mkdir(parents=True, exist_ok=True)
for d, name, sub in (("simplepir_go", "simplepir-adapter", "simplepir"), ("apir_go", "apir-adapter", "apir-code")):
    sha = git("rev-parse", "HEAD", cwd=repo / "third_party" / sub)
    sh("go", "build", "-race", "-ldflags", f"-X main.upstreamSHA={sha}", "-o", str(RACE / name), ".",
       cwd=repo / "src/adapters" / d, env={**os.environ, "GOPROXY": "off"})
sh("make", "-C", "src/adapters/vsp_cpp", f"OUT={RACE}/vsp-adapter", "EXTRA_FLAGS=-fsanitize=thread -g", cwd=repo)

run_id = "serve-threads-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw = repo / "results/raw" / run_id
raw.mkdir(parents=True)
digest = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
meta = {"run_id": run_id, "git_sha": git("rev-parse", "HEAD"), "dirty": bool(git("status", "--porcelain", "src", "Makefile")),
        "plain": {b.name: digest(b) for b in sorted(PLAIN.glob("*-adapter"))},
        "instrumented": {b.name: digest(b) for b in sorted(RACE.glob("*-adapter"))}}
(raw / "meta.json").write_text(json.dumps(meta, indent=1))
print(json.dumps(meta, indent=1))

{
 "run_id": "serve-threads-20261110T170254Z",
 "git_sha": "52afe4571c3d289fb952dafd0eb9f912c872c653",
 "dirty": true,
 "plain": {
  "apir-adapter": "2fd57ef3a448ad515d7c6c8c8ba0d6ed099cc848c8c1d86b2f4ee0f6666ea3f7",
  "simplepir-adapter": "ab8ba7a31f3808c2824b5bc95443126f00744ae5bac3b29003f7fd5aac459705",
  "vsp-adapter": "60839b1e22ccee61a08acd587577462dca87ef75baf1e4161e9105c909c147db"
 },
 "instrumented": {
  "apir-adapter": "8633f8880f873e4a663b610e3880ec4ad2e33b58eaff368d65b384d475e407a4",
  "simplepir-adapter": "a75235e803d65da4e70f9bfcd23653766741e50b48f30269d1a35b9e4a443b39",
  "vsp-adapter": "5aa8edc31cfa69451bf77526f7f53ac5d179fc4bbd5b9823d5102d1f6222ef6a"
 }
}


In [2]:
# Are the detectors live? A deliberate two-thread race, built with the same tools, must be reported.
canary = RACE / "canary"
canary.mkdir(exist_ok=True)
(canary / "race.go").write_text("""package main
import "sync"
func main() {
	n, wg := 0, sync.WaitGroup{}
	for i := 0; i < 2; i++ { wg.Add(1); go func() { defer wg.Done(); for j := 0; j < 1000; j++ { n++ } }() }
	wg.Wait()
	println(n)
}
""")
(canary / "race.cpp").write_text("""#include <thread>
int n = 0;
int main() {
	std::thread a([] { for (int j = 0; j < 100000; j++) n++; }), b([] { for (int j = 0; j < 100000; j++) n++; });
	a.join(); b.join();
	return 0;
}
""")
sh("go", "build", "-race", "-o", str(canary / "race-go"), str(canary / "race.go"), cwd=canary)
llvm = sh("brew", "--prefix", "llvm").stdout.strip()
sh(f"{llvm}/bin/clang++", "-std=c++17", "-fsanitize=thread", "-g", "-o", str(canary / "race-cpp"), str(canary / "race.cpp"))
go_out = subprocess.run([str(canary / "race-go")], capture_output=True, text=True).stderr
cpp_out = subprocess.run([str(canary / "race-cpp")], capture_output=True, text=True).stderr
print("Go race detector reported the canary:", "WARNING: DATA RACE" in go_out)
print("ThreadSanitizer reported the canary:", "WARNING: ThreadSanitizer: data race" in cpp_out)
assert "WARNING: DATA RACE" in go_out and "WARNING: ThreadSanitizer: data race" in cpp_out

Go race detector reported the canary: True
ThreadSanitizer reported the canary: True


In [3]:
# Each arm: instrumented servers with W=6 workers, 16 clients replaying a 64-query corpus (S4: 16 queries at 64 KiB)
# for 60 s with pipelining, STATS, wrong-length queries, unknown frames and mid-flight resets alongside, then
# SIGTERM under load. Clean = no race report, every answer matched its EXPECT hash, every server exited 0 with
# one summary record, none passed the 4 GiB footprint cap (the detectors add memory).
arms = [  # arm, servers, db_bytes, record_bytes, binary, corpus queries
    ("S0-go", 1, 1 << 20, 32, "simplepir-adapter", 64), ("S3", 1, 1 << 20, 0, "apir-adapter", 64),
    ("S4", 1, 1 << 16, 0, "apir-adapter", 16), ("M0", 2, 1 << 20, 32, "apir-adapter", 64),
    ("M1", 2, 1 << 20, 32, "apir-adapter", 64), ("S0-cpp", 1, 1 << 20, 32, "vsp-adapter", 64),
    ("S1", 1, 1 << 20, 32, "vsp-adapter", 64), ("S2", 1, 1 << 20, 32, "vsp-adapter", 64),
]
results = []
for arm, servers, db, rb, binary, q in arms:
    work = raw / arm
    env = {"TSAN_OPTIONS": f"log_path={work}/tsan halt_on_error=0"} if binary == "vsp-adapter" else {}
    r = serve_stress.stress(arm=arm, servers=servers, db_bytes=db, record_bytes=rb, server_bin=RACE / binary,
                            client_bin=PLAIN / binary, work=work, seconds=60, corpus_queries=q, env=env)
    results.append(r)
    tool = "TSan" if binary == "vsp-adapter" else "Go -race"
    print(f"{arm:7s} {tool:8s} matched {r['matched']:>9,}  mismatched {r['mismatched']}  race reports {r['race_reports']}  "
          f"exit {r['exit_codes']}  disturbances {r['disturbances']}  peak {max(r['peak_footprint_bytes']) / 2**20:.0f} MiB  "
          f"{'CLEAN' if r['clean'] else 'NOT CLEAN'}", flush=True)
with open(raw / "results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"\n{sum(r['matched'] for r in results):,} answers matched in total; "
      f"{sum(r['race_reports'] for r in results)} race reports; {sum(not r['clean'] for r in results)} arms not clean")
assert all(r["clean"] for r in results), [r for r in results if not r["clean"]]

S0-go   Go -race matched 1,225,754  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 60, 'wrong_length': 60, 'unknown_type': 60, 'resets': 60}  peak 151 MiB  CLEAN


S3      Go -race matched    38,950  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 60, 'wrong_length': 60, 'unknown_type': 60, 'resets': 60}  peak 776 MiB  CLEAN


S4      Go -race matched       106  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 100, 'wrong_length': 100, 'unknown_type': 100, 'resets': 100}  peak 112 MiB  CLEAN


M0      Go -race matched    31,452  mismatched 0  race reports 0  exit [0, 0]  disturbances {'stats': 118, 'wrong_length': 118, 'unknown_type': 118, 'resets': 118}  peak 69 MiB  CLEAN


M1      Go -race matched     2,176  mismatched 0  race reports 0  exit [0, 0]  disturbances {'stats': 122, 'wrong_length': 122, 'unknown_type': 122, 'resets': 122}  peak 241 MiB  CLEAN


S0-cpp  TSan     matched 1,204,546  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 60, 'wrong_length': 60, 'unknown_type': 60, 'resets': 60}  peak 106 MiB  CLEAN


S1      TSan     matched 1,201,834  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 60, 'wrong_length': 60, 'unknown_type': 60, 'resets': 60}  peak 175 MiB  CLEAN


S2      TSan     matched 1,042,778  mismatched 0  race reports 0  exit [0]  disturbances {'stats': 60, 'wrong_length': 60, 'unknown_type': 60, 'resets': 60}  peak 197 MiB  CLEAN



4,747,596 answers matched in total; 0 race reports; 0 arms not clean
